In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifier
from sklearn.feature_selection import RFE
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

# Load dataset
dataset = pd.read_csv("PA6_UL94.csv")
#dataset = pd.read_csv("PA6_update_dataset_UL_94(3.2).csv")
X = dataset.iloc[:, :-1].values  # Features
y = dataset.iloc[:, -1].values   # Target variable (classes)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=9)

# Data scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)  # Fit and transform on training data
X_test = scaler.transform(X_test)        # Transform on test data

# Load additional dataset
dataset_BDOPO_OH = pd.read_csv("3montakhab_PA6.csv")
X_BDOPO_OH = scaler.transform(dataset_BDOPO_OH)  # Apply the same scaling as training data

# Feature selection using Ridge Classifier
estimator = RidgeClassifier(alpha=1)
selector = RFE(estimator, n_features_to_select=90)
selector.fit(X_train, y_train)
X_train_selector = selector.transform(X_train)
X_test_selector = selector.transform(X_test)
X_BDOPO_OH_selector = selector.transform(X_BDOPO_OH)

# Output selected features
selected_features = selector.get_support(indices=True)
logf = open("logfile.log", "a+")
print(f"Selected feature indices: {selected_features}", file=logf, flush=True)

# Classification model
classifier = GradientBoostingClassifier(n_estimators=80, max_depth=2, random_state=1)
classifier.fit(X_train_selector, y_train)

# Model Performance Evaluation
y_train_predict = classifier.predict(X_train_selector)
y_test_predict = classifier.predict(X_test_selector)

accuracy_train = accuracy_score(y_train, y_train_predict)
accuracy_test = accuracy_score(y_test, y_test_predict)
precision = precision_score(y_test, y_test_predict, average='weighted')
recall = recall_score(y_test, y_test_predict, average='weighted')
f1 = f1_score(y_test, y_test_predict, average='weighted')

# Log metrics
print(f"Train Accuracy: {accuracy_train:.3f}", file=logf, flush=True)
print(f"Test Accuracy: {accuracy_test:.3f}", file=logf, flush=True)
print(f"Test Precision: {precision:.3f}", file=logf, flush=True)
print(f"Test Recall: {recall:.3f}", file=logf, flush=True)
print(f"Test F1 Score: {f1:.3f}", file=logf, flush=True)

# Feature Importance (optional, for interpretation)
feature_importances = classifier.feature_importances_
sorted_indices = np.argsort(feature_importances)[::-1]  # Sort in descending order

print("Top 10 Feature Importances:", file=logf, flush=True)
for idx in sorted_indices[:10]:
    print(f"Feature {selected_features[idx]}: {feature_importances[idx]:.3f}", file=logf, flush=True)

logf.close()

# Visualizing feature importances
#plt.figure(figsize=(10, 6))
#plt.bar(range(len(selected_features)), feature_importances[sorted_indices], align='center')
#plt.xticks(range(len(selected_features)), selected_features[sorted_indices], rotation=90)
#plt.xlabel('Feature Index')
#plt.ylabel('Importance')
#plt.title('Feature Importances')
#plt.tight_layout()
#plt.show()
y_BDOPO_OH_predict = classifier.predict(X_BDOPO_OH_selector)  

C:\Users\u1148893\AppData\Local\anaconda3\lib\site-packages\sklearn\base.py:443: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\u1148893\AppData\Local\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1334: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
